# mLOS — Length of Stay Analysis (Google Colab)

## Steps

### 1. Set runtime to R
- Menu: **Runtime → Change runtime type**
- Set **Runtime type** to **R**
- Click **Save**

### 2. Run Cell 1 — Setup (once per session)
This installs the required packages and creates the folder structure.
**Package installation may take a few minutes.**
Packages are only installed once per session, followed by multiple runs.

### 3. Upload your files
Use the **Files panel** (folder icon in the left sidebar) to upload your files.
After Cell 1 runs, the folders will already exist — just drop files into them.

```
/content/mLOS/
    mlos_common.R
    mlos_setup.R
    mlos_data.R
    mlos_km.R
    mlos_cox.R
    mlos_aj.R
    mlos_results.R
    mlos_excel_export.R
    mlos_render.R
    mlos_run_complete.R
    mlos_review/
        __init__.py
        __main__.py
        names.py
        bundle.py
        blocks.py
        settings.py
        output.py
        regression.py
        salience.py
        recommend.py
        figures.py
        workbook.py
        render_pptx.py
        deck.py
        variant.py
        kaplan_meier_diagram.png
        kaplan_meier_diagram.svg
    data/
        OC2_settings.yaml
        OC2_data.csv
        OC_deck_settings.yaml
```

> `mlos_review/` and `data/OC_deck_settings.yaml` are only needed for Cell 3, the
> slide deck. Leave them out and the other cells work exactly as before. The
> whole folder is needed when you do want it: every `.py` above is imported by
> the time a deck is built, so a missing one stops Cell 3 rather than costing it
> a feature.

> **Tip:** The `data/` folder and file names match your local setup exactly.  
> Cell 2 names the data and settings files in its `Sys.setenv` lines;
> edit those if your files are named differently.

### 4. Run Cell 2 — Analysis

### 5. Build the slide deck (optional)
Run **Cell 3** after Cell 2. It reads `results/results.json` and the plots
beside it and writes `reports/mlos_deck.pptx`.

The runtime stays on **R**. The deck is Python, but Colab's R runtime is an
ordinary Linux machine with Python on it, so the cell shells out rather than
asking you to switch runtimes and re-run the analysis. Nothing about the deck
touches R.

### 6. Download results
Results are saved to `/content/mLOS/results/`, and a deck to `/content/mLOS/reports/`.  
Right-click any file in the Files panel and choose **Download**,  
or run Cell 4 to package everything as a zip, then download `mlos_results.zip` from the Files panel.


In [ ]:
# Cell 1 — Setup (run once per session)
# Creates the folder structure and installs required packages.
# Package installation may take a few minutes.

dir.create("/content/mLOS/data",        recursive = TRUE, showWarnings = FALSE)
dir.create("/content/mLOS/mlos_review", recursive = TRUE, showWarnings = FALSE)
dir.create("/content/mLOS/results",     recursive = TRUE, showWarnings = FALSE)
dir.create("/content/mLOS/reports",     recursive = TRUE, showWarnings = FALSE)
cat("Folders ready.\n")

install.packages(c("survival", "yaml", "jsonlite", "openxlsx"), quiet = TRUE)
# flexsurv enables the Weibull regression (parametric_regression: WEIBULL)
install.packages("flexsurv", quiet = TRUE)
# The Python packages are for the optional slide deck in Cell 3, and are
# installed here so that there is one place a session installs anything.
# pandas and matplotlib are usually already in Colab's image, and pip is a
# no-op when they are. This list is checked against pyproject.toml by the test
# suite, which is where the requirements are declared; it is spelled out here
# rather than installed from that file because a session may not have had
# pyproject.toml uploaded to it.
system("pip install --quiet python-pptx pyyaml pandas matplotlib openpyxl")
cat("Packages installed.\n")

In [ ]:
# Cell 2 — Run the analysis
# Upload your files first (see Step 3 above).
#
# These name your data and settings files (paths are relative to
# /content/mLOS); edit them if your files are named differently.
Sys.setenv(MLOS_DATA_FILE     = "data/OC2_data.csv")
Sys.setenv(MLOS_SETTINGS_FILE = "data/OC2_settings.yaml")
setwd("/content/mLOS")
source("mlos_run_complete.R")

In [ ]:
# Cell 3 - Build the slide deck (optional; run after Cell 2)
#
# The runtime is R, so this hands the work to the Python that Colab's image
# also carries. The deck reads results/results.json and the PNGs beside it and
# touches no R at all, which is why it can be a shell-out rather than a
# runtime change. The Python packages it needs were installed by Cell 1.
#
# Settings are read from data/OC_deck_settings.yaml if you uploaded one; without it
# the deck runs on defaults. An earlier deck at the same name is archived, not
# overwritten.
setwd("/content/mLOS")

status <- system("python3 -m mlos_review.deck results")

if (status == 0) {
  cat("\nDeck written to reports/. Download it from the Files panel,\n")
  cat("or run Cell 4 to include it in the zip.\n")
} else {
  cat("\nDeck build failed (exit ", status, "). The analysis results in\n", sep = "")
  cat("results/ are unaffected.\n")
}


In [ ]:
# Cell 4 - Zip results for download
# Only the current run is packaged. Earlier runs are archived under
# results/mLOS_<date>_<n>/ and are left out of the zip; delete them from the
# Files panel when you no longer need them.
#
# reports/ is added whole when it exists: a deck is small, and the archived
# copies of earlier decks are usually what you want alongside the current one.
current <- list.files("results", full.names = TRUE)
current <- current[!grepl("/mLOS_[0-9]{8}_[0-9]+$", current)]
if (dir.exists("reports")) current <- c(current, "reports")
zip("/content/mLOS/mlos_results.zip", files = current)
